# Skin Lesion — Complete Per-Image Results, No XAI

**Kaggle P100 · Save Version → Save & Run All · automatic ZIP export**

This notebook supplements the existing results identified in your 13-page Per-Image Result Generation Plan. It does not retrain or reselect your final model. It creates full per-image numerical records and offline reports for internal validation/test, official ISIC test, PAD zero-shot/restricted/adapted, HAM, and training data. It also extends rotation/pair diagnostics and t-SNE to every internal test image. No XAI code runs.

## Attach these SAVED NOTEBOOK OUTPUTS before running

1. **last-part-of-the-previous-one** — REQUIRED entire saved output; includes `rotation_ssl_paperlike`, weights, caches and original images if retained.
2. **confidence-gated-ensemble** — REQUIRED entire saved output, including internal, external and PAD meta-adapted folders.
3. **complete-result-generator-ipynb** — optional; preserves the existing rich internal master and numerical decomposition.
4. **ham10000-all-model-evaluation** — optional; adds the full HAM cached evaluation.
5. **output-result-for-pad-fusion** — optional; adds feature-level PAD-adapted records.
6. **supervised-classification-for-journal** — optional; adds original-split baseline probabilities and can recover the previously unsaved supervised HAM probabilities when HAM is also attached.
7. **journal-extension-j2** and **dinov2-final** — optional DINOv2 outputs, kept under their correct separate protocols.

Use Kaggle **Add Input → Notebook outputs**. Attaching these saved outputs is sufficient; do not upload the `.ipynb` source files as datasets. Attach only ONE saved version of each output, retaining its folder structure. There is no need to download outputs locally and upload them again.

## Image inputs for photographs in every report

Attach the **same official ISIC 2019 test image dataset and PAD-UFES-20 image dataset used by your existing notebooks**, plus **Skin Cancer MNIST: HAM10000** (`HAM10000_images_part_1` and `HAM10000_images_part_2`). Their names/paths are detected automatically. Already attached image copies are reused. The original resized internal ISIC dataset downloads automatically from the original Hugging Face URL only if needed.

Missing external photographs do **not** discard cached numerical results: the report states that the photograph is unavailable, `coverage.csv` records the count, and the ZIP is explicitly labelled as incomplete photograph coverage. The notebook never substitutes an unrelated image with a similar numeric suffix.

## Run once

Select **GPU P100**, enable Internet, then **Save Version → Save & Run All**. Run all cells in order; no flags or paths need editing. Read `COMPLETION.json` and download the ZIP from the last cell or the saved version's Output tab. Extract the ZIP and open `Skin_Lesion_Per_Image_No_XAI/index.html`.

The saved source is under 1 MB and contains no embedded notebook output images. GPU execution and access to your actual saved Kaggle artifacts cannot be tested in this authoring environment. The code performs runtime identity, probability, coverage and reconstruction checks and creates a clearly labelled diagnostic ZIP if a stage fails.

### Corrections to the PDF plan

- The mandatory total is **15,603** (2,533 + 2,534 + 8,238 + 2,298), not 13,410. Including training and HAM gives **45,882**, before extra restricted/adapted representations of the same PAD images.
- SimCLR No-TTA training predictions already exist in `multiseed_probabilities_no_tta.npz`; they are reused.
- The supervised HAM probabilities were computed in memory but not saved; an added GPU stage recovers them from the five saved baseline checkpoints.
- HAM's zero full-string-ID matches do not prove image independence after renaming.
- Some legacy NPZ caches contain labels but no image IDs. Their original split order is preserved and checked, with this provenance limitation recorded.
- B10 refers to a different final-archive notebook that was not attached. This deliverable creates the non-XAI result supplement; it does not claim to be the full thesis submission archive.


## Input and per-image reporting utilities

In [1]:
# Shared input, validation and per-image reporting functions.
import os, sys, re, gc, json, math, hashlib, shutil, zipfile, traceback, platform, html
from pathlib import Path
from collections import OrderedDict, defaultdict
import numpy as np
import pandas as pd
import joblib
from PIL import Image
from IPython.display import display, FileLink
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ROOT = WORK / 'Skin_Lesion_Per_Image_No_XAI'
if ROOT.exists():
    shutil.rmtree(ROOT)  # Clear only this notebook's own previous generated output.
ROOT.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
MODEL_NAMES = ['SimCLR_NoTTA','SimCLR_SelectedTTA','Rotation','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
SEED, IMG_SIZE, NUM_CLASSES, EVAL_BATCH = 42,224,8,32
STATUS, PROVENANCE, COVERAGE, METRICS, GALLERIES = [],[],[],[],[]
IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}

def save_json(value, path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,default=str,allow_nan=False),encoding='utf-8')

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()

def find_one(suffix, required=True):
    hits=[p for p in FILES if p.as_posix().endswith(suffix)]
    if not hits:
        if required: raise FileNotFoundError('Attach the saved output containing: '+suffix)
        return None
    hits=sorted(hits,key=lambda p:(len(p.parts),str(p)))
    if len(hits)>1:
        # Prefer canonical saved-notebook output roots. Copied evidence files may also be attached.
        preferred=[p for p in hits if any(x in p.parts for x in ['rotation_ssl_paperlike','reuse_only_all_model_ensemble','both_stacking_models_external_evaluation','pad_domain_adapted_both_stacking_models','PAD_ALL_MODELS_ADAPTED','mbec_ham10000_all_models','mbec_plain_supervised_resnet50'])]
        if preferred:hits=preferred
        hashes={sha(p) for p in hits}
        if len(hashes)>1:raise ValueError('Conflicting saved versions: '+str(hits)+'. Attach only one saved version of this output.')
    p=hits[0];PROVENANCE.append({'path':str(p),'sha256':sha(p),'role':suffix});return p

def normp(p):
    p=np.asarray(p,dtype=np.float64)
    if p.ndim!=2 or not np.isfinite(p).all() or (p<0).any() or (p.sum(1)<=0).any():
        raise ValueError('Invalid probability matrix; no NaN/negative/empty row repair is allowed.')
    if not np.allclose(p.sum(1),1,atol=2e-4):raise ValueError('Rows are not probabilities.')
    return (p/p.sum(1,keepdims=True)).astype(np.float32)

def idcol(df):
    for c in ['image_id','image','img_id','isic_id','filename','file_name','filepath','image_path','resolved_filepath','path']:
        if c in df:return c
    raise ValueError('No image identifier column: '+str(list(df.columns)))

def stem(v):return Path(str(v).replace('\\','/')).stem

def labels_of(df):
    for c in ['y_true_6','true_label','y_true','label','class_id','target']:
        if c in df:return pd.to_numeric(df[c],errors='coerce').fillna(-1).to_numpy(np.int32)
    raise ValueError('No ground truth column')

def pcols(df, names=CLASS_NAMES):
    # The supplied notebooks save several valid naming variants, for example
    # prob_class_0, prob_0_MEL, prob_MEL, class_0_prob and p0.
    # Resolve names by class index first, rather than assuming one notebook's
    # convention for every saved prediction CSV.
    normalized={str(c).strip().lower():c for c in df.columns}
    out=[]
    indexed={}
    patterns=[
        r'^prob_class_(\d+)$', r'^probability_class_(\d+)$',
        r'^prob_(\d+)(?:_[a-z0-9]+)?$', r'^probability_(\d+)$',
        r'^class_(\d+)_prob(?:ability)?$', r'^p_?(\d+)$',
    ]
    for column in df.columns:
        text=str(column).strip().lower()
        for pattern in patterns:
            match=re.match(pattern,text)
            if match:
                index=int(match.group(1))
                if index in indexed and indexed[index]!=column:
                    raise ValueError(f'Ambiguous probability columns for class {index}: {indexed[index]!r}, {column!r}')
                indexed[index]=column
                break
    for i,n in enumerate(names):
        candidates=[
            normalized.get(f'prob_{i}_{n}'.lower()), normalized.get(f'prob_class_{i}'),
            normalized.get(f'probability_class_{i}'), normalized.get(f'prob_{n}'.lower()),
            normalized.get(f'p_{n}'.lower()), normalized.get(f'probability_{n}'.lower()),
            indexed.get(i),
        ]
        column=next((value for value in candidates if value is not None),None)
        if column is None:
            available=[str(c) for c in df.columns if any(word in str(c).lower() for word in ['prob','score','softmax'])]
            raise ValueError(f'Missing probability for {n}. Available probability-like columns: {available}')
        out.append(column)
    if len(set(out))!=len(out):raise ValueError('One CSV column was selected for more than one class.')
    return out

def frame_of(df, dataset, usage, names=CLASS_NAMES):
    d=df.copy().reset_index(drop=True);d['image_id']=d[idcol(d)].map(stem)
    if d.image_id.duplicated().any():raise ValueError('Duplicate image IDs within '+dataset)
    d['label']=labels_of(d);d['is_labelled']=d.label.between(0,len(names)-1)
    d['class_name']=[names[y] if 0<=y<len(names) else None for y in d.label]
    d['dataset']=dataset;d['split_usage']=usage;d['sample_uid']=dataset+'__'+d.image_id
    return d

def align_csv(path, frame, names=CLASS_NAMES):
    d=pd.read_csv(path);p=normp(d[pcols(d,names)].to_numpy());y=labels_of(d)
    try:c=idcol(d)
    except ValueError:c=None
    if c:
        ids=d[c].map(stem)
        if ids.duplicated().any() or set(ids)!=set(frame.image_id):raise ValueError('Image identity mismatch: '+str(path))
        order=pd.Index(ids).get_indexer(frame.image_id);p=p[order];y=y[order];mode='image_id join'
    else:mode='legacy positional order; labels checked; source contains no IDs'
    if len(p)!=len(frame) or not np.array_equal(y,frame.label.to_numpy()):raise ValueError('Label/order mismatch: '+str(path))
    PROVENANCE.append({'path':str(path),'alignment':mode})
    return p

def align_npz(z, pk, yk, frame, idkey=None):
    p=normp(z[pk]);y=np.asarray(z[yk],np.int32)
    if idkey and idkey in z:
        ids=np.asarray([stem(x) for x in z[idkey]])
        if len(set(ids))!=len(ids) or set(ids)!=set(frame.image_id):raise ValueError('NPZ IDs differ')
        order=pd.Index(ids).get_indexer(frame.image_id);p=p[order];y=y[order]
    if len(p)!=len(frame) or not np.array_equal(y,frame.label.to_numpy()):raise ValueError('NPZ label/order mismatch: '+pk)
    return p

def add_block(frame, prefix, p, names=CLASS_NAMES):
    p=normp(p);n=len(frame)
    if p.shape!=(n,len(names)):raise ValueError('Probability/class shape mismatch')
    y=frame.label.to_numpy(int); valid=(y>=0)&(y<len(names));idx=np.flatnonzero(valid)
    pred=p.argmax(1);ordered=np.sort(p,axis=1);truth=np.full(n,np.nan);rank=truth.copy();correct=truth.copy()
    truth[idx]=p[idx,y[idx]]
    # Competition rank: 1 + number of probabilities strictly greater than the true-class probability.
    rank[idx]=1+(p[idx]>truth[idx,None]).sum(1);correct[idx]=(pred[idx]==y[idx]).astype(int)
    return pd.DataFrame({f'{prefix}_predicted_class_id':pred,f'{prefix}_predicted_class_name':np.asarray(names)[pred],
        f'{prefix}_confidence':p.max(1),f'{prefix}_margin':ordered[:,-1]-ordered[:,-2],
        f'{prefix}_entropy':-(p*np.log(np.clip(p,1e-12,1))).sum(1),f'{prefix}_true_class_probability':truth,
        f'{prefix}_true_class_rank':rank,f'{prefix}_correct':correct,
        **{f'{prefix}_prob_{name}':p[:,i] for i,name in enumerate(names)}})

IMAGE_INDEX=defaultdict(list)
def index_images(roots):
    for root in roots:
        for p in root.rglob('*'):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:IMAGE_INDEX[p.stem].append(p)

def resolve_image(row):
    for c in ['filepath','resolved_filepath','image_path','path','original_filepath']:
        v=row.get(c)
        if isinstance(v,str) and Path(v).is_file():return Path(v)
    hits=list(dict.fromkeys(IMAGE_INDEX.get(str(row['image_id']),[])))
    # Exact full IDs only: never interpret ISIC_Resize_N as ISIC_N.
    if len(hits)==1:return hits[0]
    if len(hits)>1:
        hashes={sha(p) for p in hits}
        if len(hashes)==1:return hits[0]
        # Prefer the source family matching original stored path when replicas differ in resolution.
        original=' '.join(str(row.get(c,'')) for c in ['filepath','original_filepath'])
        hint=next((s for s in ['HAM10000','ISIC_2019_Test','ISIC_Resize','pad-ufes','PAD-UFES'] if s.lower() in original.lower()),None)
        narrowed=[p for p in hits if hint and hint.lower() in str(p).lower()]
        if len(narrowed)==1:return narrowed[0]
        return None
    return None

def export_dataset(frame, streams, names=CLASS_NAMES, base_names=None, final='FinalEnsemble'):
    dataset=str(frame.dataset.iloc[0]);folder=ROOT/'01_PER_IMAGE_RESULTS'/dataset;folder.mkdir(parents=True,exist_ok=True)
    if not streams:raise ValueError('No models for '+dataset)
    master=pd.concat([frame.reset_index(drop=True)]+[add_block(frame,m,p,names) for m,p in streams.items()],axis=1)
    bases=base_names if base_names is not None else list(streams)
    bp=np.stack([streams[m].argmax(1) for m in bases],axis=1)
    master['model_agreement_count']=np.asarray([np.unique(row,return_counts=True)[1].max() for row in bp])
    if final in streams:
        fp=streams[final].argmax(1);bc=bp==frame.label.to_numpy()[:,None];fc=fp==frame.label.to_numpy()
        master['models_agreeing_with_final']=(bp==fp[:,None]).sum(1)
        master['case_category']=np.select([~frame.is_labelled.to_numpy(),fc&bc.all(1),fc&~bc.any(1),fc,~fc&bc.any(1)],
            ['unlabelled','all_models_correct','final_rescued_all_base_wrong','final_correct_mixed_base_results','final_wrong_but_some_base_correct'],default='all_models_wrong')
    else:master['case_category']='no_designated_final_model'
    master.to_csv(folder/'all_image_results.csv',index=False)
    np.savez_compressed(folder/'probabilities.npz',image_ids=frame.image_id.to_numpy(dtype=str),labels=frame.label.to_numpy(),class_names=np.asarray(names),**{f'prob__{m}':p for m,p in streams.items()})
    # Recompute evaluation metrics only on labelled, non-training rows.
    keep=frame.is_labelled.to_numpy() & ~frame.split_usage.str.contains('in_sample').to_numpy()
    for model,p in streams.items():
        if keep.any():
            y=frame.label.to_numpy()[keep];q=p[keep];pred=q.argmax(1)
            row={'dataset':dataset,'split_usage':frame.split_usage.iloc[0],'model':model,'n':len(y),'accuracy':accuracy_score(y,pred),'bacc_present_classes':balanced_accuracy_score(y,pred),'macro_f1_fixed_classes':f1_score(y,pred,labels=range(len(names)),average='macro',zero_division=0)}
            row['macro_auc_ovr']=roc_auc_score(y,q,labels=range(len(names)),multi_class='ovr',average='macro') if len(np.unique(y))==len(names) else None
            METRICS.append(row)
            pd.DataFrame(confusion_matrix(y,pred,labels=range(len(names))),index=names,columns=names).to_csv(folder/(model+'_confusion_matrix.csv'))
    # Offline HTML has one numerical JSON record and one page per image; all pages share compact thumbnails.
    pages=folder/'html';records=folder/'json';thumbs=folder/'images'
    for p in [pages,records,thumbs]:p.mkdir(exist_ok=True)
    entries=[];found=0
    for i,row in master.iterrows():
        uid=hashlib.sha256(str(row.sample_uid).encode()).hexdigest()[:24]
        record=json.loads(row.to_json());save_json(record,records/(uid+'.json'))
        source=resolve_image(row);image_part='<p>Source image is not attached or its identity is ambiguous. Numerical result is available.</p>'
        if source:
            try:
                with Image.open(source) as im:
                    im=im.convert('RGB');im.thumbnail((320,320));im.save(thumbs/(uid+'.jpg'),quality=85)
                found+=1;image_part=f'<img src="../images/{uid}.jpg" alt="{html.escape(str(row.image_id))}">'
            except Exception as e:image_part='<p>Image could not be decoded: '+html.escape(str(e))+'</p>'
        rows=[]
        for m in streams:
            correct=row[f'{m}_correct']; verdict='unlabelled' if pd.isna(correct) else ('correct' if correct else 'incorrect')
            probs='; '.join(f'{c}: {row[f"{m}_prob_{c}"]:.6f}' for c in names)
            rows.append(f'<tr><td>{html.escape(m)}</td><td>{row[f"{m}_predicted_class_name"]}</td><td>{row[f"{m}_confidence"]:.6f}</td><td>{verdict}</td><td>{probs}</td></tr>')
        text=f'<!doctype html><meta charset="utf-8"><style>body{{font:16px Arial;margin:30px;max-width:1300px}}td,th{{padding:8px;border:1px solid #ddd}}table{{border-collapse:collapse}}img{{max-width:320px}}</style><a href="../index.html">Back to image index</a><h1>{html.escape(str(row.image_id))}</h1><p>{html.escape(dataset)} | {html.escape(str(row.split_usage))} | True class: {html.escape(str(row.class_name))}</p>{image_part}<table><tr><th>Model</th><th>Prediction</th><th>Confidence</th><th>Correctness</th><th>Class probabilities</th></tr>{"".join(rows)}</table><p><a href="../json/{uid}.json">Full numerical record (margin, entropy, rank and probabilities)</a></p>'
        (pages/(uid+'.html')).write_text(text,encoding='utf-8')
        entries.append(f'<tr><td><a href="html/{uid}.html">{html.escape(str(row.image_id))}</a></td><td>{html.escape(str(row.class_name))}</td><td>{html.escape(str(row.case_category))}</td></tr>')
        if (i+1)%2000==0:print(dataset,'reports',i+1,'/',len(master),flush=True)
    (folder/'index.html').write_text('<!doctype html><meta charset="utf-8"><h1>'+html.escape(dataset)+'</h1><input placeholder="Search image/class/category" oninput="for(const r of document.querySelectorAll(\'tbody tr\'))r.hidden=!r.textContent.toLowerCase().includes(this.value.toLowerCase())"><p><a href="all_image_results.csv">Download CSV</a></p><table><tbody>'+''.join(entries)+'</tbody></table>',encoding='utf-8')
    COVERAGE.append({'dataset':dataset,'images':len(frame),'models':len(streams),'labelled':int(frame.is_labelled.sum()),'html_reports':len(entries),'images_displayed':found,'all_images_displayed':found==len(frame),'usage':frame.split_usage.iloc[0]})
    GALLERIES.append((dataset,str((folder/'index.html').relative_to(ROOT))))
    print('Exported',dataset,len(frame),'images,',len(streams),'models; photographs',found,flush=True)
    return master


## Exact saved-data readers

In [2]:
def prepare_inputs():
    global FILES,PIPELINE_ROOT,ENSEMBLE_ROOT,FINAL_CONFIG,STACKER,SPLITS
    FILES=[p for p in INPUT.rglob('*') if p.is_file()]
    marker=find_one('rotation_ssl_paperlike/splits/rotation_fixed_train_val_test_split.json',required=False)
    if marker is None:
        fallback=find_one('splits/rotation_fixed_train_val_test_split.json',required=False)
        if fallback is None:raise FileNotFoundError('The saved OUTPUT of last-part-of-the-previous-one is missing. Adding its .ipynb source file is not sufficient.')
        marker=fallback
    # The pipeline root is the parent containing the splits directory.
    PIPELINE_ROOT=marker.parent.parent
    config=find_one('reuse_only_all_model_ensemble/final_ensemble_config.json',required=False)
    if config is None:raise FileNotFoundError('The saved OUTPUT of confidence-gated-ensemble is missing. Adding its .ipynb source file is not sufficient.')
    ENSEMBLE_ROOT=config.parent;FINAL_CONFIG=json.loads(config.read_text());STACKER=ENSEMBLE_ROOT/'stacking_meta_model.joblib'
    if FINAL_CONFIG['models']!=MODEL_NAMES:raise ValueError('Frozen source order changed')
    split=json.loads(marker.read_text());SPLITS={}
    for key,n in [('train',20264),('val',2533),('test',2534)]:
        d=frame_of(pd.DataFrame(split[key]),'ISIC2019_'+key,'train_in_sample' if key=='train' else ('validation_model_selection' if key=='val' else 'internal_test'))
        if len(d)!=n:raise ValueError(f'Locked {key} size is {len(d)}; expected {n}. No new split will be generated.')
        SPLITS[key]=d
    allids=pd.concat([f.image_id for f in SPLITS.values()])
    if allids.duplicated().any():raise ValueError('Locked split has repeated IDs; inspect source before proceeding')
    (ROOT/'00_AUDIT').mkdir(exist_ok=True)
    shutil.copy2(config,ROOT/'00_AUDIT'/'final_ensemble_config.json')
    index_images([INPUT])
    print('Frozen split restored:',{k:len(v) for k,v in SPLITS.items()})

def calibrated_ensemble(sources):
    def temp(p,t):
        a=np.log(np.clip(normp(p),1e-12,1))/float(t);a-=a.max(1,keepdims=True);a=np.exp(a);return normp(a/a.sum(1,keepdims=True))
    cal=OrderedDict((n,temp(sources[n],FINAL_CONFIG['temperatures'][n])) for n in MODEL_NAMES)
    weight=sum(float(FINAL_CONFIG['optimized_weights'][n])*cal[n] for n in MODEL_NAMES);weight=normp(weight)
    blocks=[]
    for p in cal.values():blocks.extend([p,np.log(np.clip(p,1e-12,1)),p.max(1,keepdims=True),-(p*np.log(np.clip(p,1e-12,1))).sum(1,keepdims=True)])
    stacker=joblib.load(STACKER);raw=stacker.predict_proba(np.concatenate(blocks,axis=1).astype(np.float32))
    classes=stacker.named_steps['classifier'].classes_ if hasattr(stacker,'named_steps') else stacker.classes_
    ps=np.zeros((len(raw),8),np.float32)
    for j,c in enumerate(classes):ps[:,int(c)]=raw[:,j]
    ps=normp(ps);a=float(FINAL_CONFIG['stacking_alpha']);blend=normp((1-a)*cal[FINAL_CONFIG['anchor']]+a*ps)
    z=np.log(np.clip(blend,1e-12,1))+np.asarray(FINAL_CONFIG['class_bias'])[None,:];z-=z.max(1,keepdims=True);q=np.exp(z);final=normp(q/q.sum(1,keepdims=True))
    streams=OrderedDict(sources);streams.update(OptimizedWeighted=weight,StackingBlend=blend,StackingClassBias=final,FinalEnsemble=final)
    return streams

def internal_results():
    global INTERNAL_SOURCES
    INTERNAL_SOURCES={}
    for split in ['val','test']:
        frame=SPLITS[split];streams=OrderedDict()
        for m,spec in MODEL_PATHS_SPEC.items():
            if spec['type']=='csv':p=align_csv(spec[split],frame)
            else:
                with np.load(spec['path'],allow_pickle=False) as z:p=align_npz(z,split+'_prob','y_'+split,frame)
                PROVENANCE.append({'path':str(spec['path']),'alignment':'legacy positional split order; labels verified; NPZ has no image IDs'})
            streams[m]=p
        INTERNAL_SOURCES[split]=OrderedDict(streams)
        with np.load(ENSEMBLE_ROOT/'final_internal_probabilities.npz',allow_pickle=False) as z:
            for name,key in [('OptimizedWeighted','weighted'),('StackingBlend','stacking'),('StackingClassBias','stacking_bias'),('FinalEnsemble','final')]:
                streams[name]=align_npz(z,f'{key}_{split}_prob','y_'+split,frame)
        # Validation remains the stored OOF/selection result. Never feed validation through its fitted stacker.
        if split=='test':
            reconstructed=calibrated_ensemble(INTERNAL_SOURCES[split])['FinalEnsemble']
            delta=float(np.abs(reconstructed-streams['FinalEnsemble']).max())
            save_json({'max_absolute_probability_error':delta,'tolerance':2e-5},ROOT/'00_AUDIT'/'ensemble_reconstruction.json')
            if delta>2e-5:raise ValueError('Frozen stacker does not reproduce saved final test probabilities')
        for suffix,name in [('supervised_resnet50_5seed_probabilities.npz','SupervisedResNet50_5Seed'),('dinov2_original_split_multiseed_probabilities.npz','DINOv2_OriginalSplit_3Seed')]:
            file=find_one(suffix,required=False)
            if file:
                with np.load(file,allow_pickle=True) as z:streams[name]=align_npz(z,split+'_prob','y_'+split,frame,split+'_image_id')
        export_dataset(frame,streams,base_names=MODEL_NAMES)

def external_results():
    for dataset,expected in [('ISIC_2019_Official_Test',8238),('PAD_UFES_20_External',2298)]:
        p=find_one('both_stacking_models_external_evaluation/'+dataset+'/SimCLR_NoTTA/predictions.csv')
        frame=frame_of(pd.read_csv(p),dataset,'external_evaluation');streams=OrderedDict()
        if len(frame)!=expected:raise ValueError('External image count mismatch')
        for m in MODEL_NAMES+['NoTTA_StackingBlend','NoTTA_StackingBlend_ClassBias']:
            streams[m]=align_csv(p.parent.parent/m/'predictions.csv',frame)
        streams['FinalEnsemble']=streams['NoTTA_StackingBlend_ClassBias']
        export_dataset(frame,streams,base_names=MODEL_NAMES)
        if dataset.startswith('PAD'):
            ids=[0,1,2,3,4,7];names=[CLASS_NAMES[i] for i in ids];d=frame.copy()
            d=d.drop(columns=[c for c in ['y_true','true_label','y_true_6'] if c in d],errors='ignore');d['label']=d.label.map({v:i for i,v in enumerate(ids)});d=frame_of(d,'PAD_UFES20_Restricted6','external_evaluation',names)
            # frame_of must use remapped labels; source CSV has no y_true_6, retain only canonical columns.
            d=d.drop(columns=[c for c in ['y_true','true_label','y_true_6'] if c in d],errors='ignore')
            restricted=OrderedDict()
            for m,p in streams.items():
                q=p[:,ids];restricted[m]=normp(q/q.sum(1,keepdims=True))
            export_dataset(d,restricted,names,MODEL_NAMES)

def ham_results():
    p=find_one('mbec_ham10000_all_models/base_probability_cache/all_seven_sources.npz')
    with np.load(p,allow_pickle=True) as z:
        frame=frame_of(pd.DataFrame({'image_id':z['image_ids'].astype(str),'label':z['labels']}),'HAM10000','overlap_unverified_secondary_analysis')
        if len(frame)!=10015:raise ValueError('HAM count mismatch')
        if z['model_names'].astype(str).tolist()!=MODEL_NAMES:raise ValueError('HAM class source order mismatch')
        sources=OrderedDict((m,normp(z['prob__'+m])) for m in MODEL_NAMES)
    frame['overlap_status']='unverified_after_possible_image_renaming'
    export_dataset(frame,calibrated_ensemble(sources),base_names=MODEL_NAMES)
    save_json({'status':'unverified','reason':'Original HAM code compares full identifier strings. ISIC_Resize names cannot establish image-level independence from HAM. No leakage-free claim is made here.','scope':'full 10015-image cached evaluation'},ROOT/'00_AUDIT'/'HAM_overlap_limit.json')

# Preserve separate PAD adaptation protocols and their native six-class order.
def pad_adapted_results():
    found=False
    for rootname in ['PAD_ALL_MODELS_ADAPTED','pad_domain_adapted_both_stacking_models']:
        roots=[]
        for p in FILES:
            if rootname in p.parts:
                root=Path(*p.parts[:p.parts.index(rootname)+1])
                if root not in roots:roots.append(root)
        if len(roots)>1:raise ValueError('Multiple adapted output roots: '+str(roots))
        if not roots:continue
        found=True;root=roots[0]
        for split in ['train','val','validation','test']:
            paths=sorted(root.rglob(split+'_predictions.csv'))
            if not paths:continue
            groups={}
            for p in paths:
                d=pd.read_csv(p)
                # Native class names are recorded directly in ordered probability-column suffixes.
                cols=sorted([c for c in d if re.match(r'^prob_\d+_',c)],key=lambda c:int(c.split('_')[1]))
                names=[c.split('_',2)[2] for c in cols]
                if len(names) not in (6,8):continue
                group=tuple(names);groups.setdefault(group,[]).append(p)
            for names,paths in groups.items():
                first=pd.read_csv(paths[0]);frame=frame_of(first,rootname+'_'+split,'train_in_sample' if split=='train' else ('validation_model_selection' if split in ['val','validation'] else 'adapted_test'),list(names))
                streams=OrderedDict((re.sub(r'[^A-Za-z0-9_]+','_',str(p.parent.relative_to(root))),align_csv(p,frame,list(names))) for p in paths)
                export_dataset(frame,streams,list(names),final='no_designated_final')
    if not found:raise FileNotFoundError('Attach output-result-for-pad-fusion and confidence-gated-ensemble saved outputs for adapted PAD records')
    # Meta-adapted validation predictions are stored in an NPZ, not in per-method CSVs.
    meta=find_one('pad_domain_adapted_both_stacking_models/pad_domain_adapted_probabilities.npz',required=False)
    if meta:
        cfg=json.loads((meta.parent/'pad_domain_adaptation_config.json').read_text());names=cfg['common_classes']
        full=pd.read_csv(meta.parent/'pad_domain_adaptation_split.csv');d=full[full['split'].isin(['val','validation'])].copy().reset_index(drop=True)
        d['y_true_6']=d['label_6'];frame=frame_of(d,'PAD_MetaAdapted_validation','validation_model_selection',names)
        with np.load(meta) as z:
            streams=OrderedDict((name,align_npz(z,key,'y_val',frame)) for name,key in [('StackingBlend','no_bias_val_prob'),('StackingClassBias','class_bias_val_prob')])
        export_dataset(frame,streams,names,final='no_designated_final')


def grouped_dino_results():
    paths=[p for p in FILES if p.name.endswith('_dinov2_features_and_probabilities.npz') and 'q2_dinov2_final' in p.parts]
    if not paths:
        STATUS.append({'stage':'Separate grouped-split DINOv2','status':'NOT_ATTACHED','reason':'Optional separate experiment; never merged with original split.'});return
    for p in paths:
        with np.load(p,allow_pickle=True) as z:
            frame=frame_of(pd.DataFrame({'image_id':z['image_id'],'label':z['labels']}),'DINO_GROUPED_'+p.name.split('_dinov2')[0],'grouped_protocol_separate')
            export_dataset(frame,OrderedDict(DINOv2_Grouped=normp(z['probabilities'])),final='no_designated_final')

def pad_meta_results():
    meta=find_one('pad_domain_adapted_both_stacking_models/pad_domain_adapted_probabilities.npz')
    cfg=json.loads((meta.parent/'pad_domain_adaptation_config.json').read_text());names=cfg['common_classes']
    full=pd.read_csv(meta.parent/'pad_domain_adaptation_split.csv')
    for split,key in [('validation','val'),('test','test')]:
        d=full[full['split'].isin(['val','validation'] if split=='validation' else ['test'])].copy().reset_index(drop=True)
        d['y_true_6']=d['label_6'];frame=frame_of(d,'PAD_MetaAdapted_'+split,'validation_model_selection' if split=='validation' else 'adapted_test',names)
        with np.load(meta) as z:
            streams=OrderedDict((name,align_npz(z,f'{prefix}_{key}_prob','y_'+key,frame)) for name,prefix in [('StackingBlend','no_bias'),('StackingClassBias','class_bias')])
        export_dataset(frame,streams,names,final='StackingClassBias')


## GPU inference and full-image diagnostics

In [3]:
def init_gpu():
    global tf,keras,layers,ResNet50
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.applications import ResNet50
    devices=tf.config.list_physical_devices('GPU')
    if not devices:raise RuntimeError('GPU missing. Select GPU P100 before Save & Run All.')
    for gpu in devices:tf.config.experimental.set_memory_growth(gpu,True)
    tf.keras.utils.set_random_seed(SEED)
    # P100: float32; no bfloat16 and no package/environment replacement.
    keras.mixed_precision.set_global_policy('float32')
    save_json({'python':sys.version,'tensorflow':tf.__version__,'numpy':np.__version__,'devices':[str(x) for x in devices]},ROOT/'00_AUDIT'/'environment.json')
    print('TensorFlow',tf.__version__,devices)

def ensure_internal_images():
    missing=[row.image_id for _,row in pd.concat(list(SPLITS.values())).iterrows() if resolve_image(row) is None]
    if missing:
        import requests
        directory=WORK/'no_xai_downloaded_isic';directory.mkdir(exist_ok=True);archive=directory/'data.zip'
        url='https://huggingface.co/datasets/MKZuziak/ISIC_2019_224/resolve/main/data.zip'
        print('Downloading original resized ISIC dataset because',len(missing),'images are absent.',flush=True)
        if not archive.exists():
            partial=archive.with_suffix('.partial')
            with requests.get(url,stream=True,timeout=(30,180)) as response:
                response.raise_for_status()
                with open(partial,'wb') as f:
                    for b in response.iter_content(2**20):f.write(b)
            partial.replace(archive)
        with zipfile.ZipFile(archive) as z:
            for info in z.infolist():
                destination=(directory/info.filename).resolve()
                if not destination.is_relative_to(directory.resolve()):raise ValueError('Unsafe ZIP member')
                if not destination.exists():z.extract(info,directory)
        index_images([directory])
    for k,d in SPLITS.items():
        paths=[resolve_image(row) for _,row in d.iterrows()]
        if any(p is None for p in paths):raise FileNotFoundError('Exact original image identities could not all be resolved for '+k)
        d['filepath']=[str(p) for p in paths]

def make_ds(df):
    def decode(path,label):
        x=tf.io.read_file(path);x=tf.io.decode_image(x,channels=3,expand_animations=False)
        x=tf.image.resize(x,[224,224]);x=tf.cast(x,tf.float32)/255.;x.set_shape([224,224,3]);return x,label
    ds=tf.data.Dataset.from_tensor_slices((df.filepath.to_numpy(dtype=str),df.label.to_numpy(np.int32)))
    options=tf.data.Options();options.experimental_deterministic=True
    return ds.with_options(options).map(decode,num_parallel_calls=tf.data.AUTOTUNE).batch(EVAL_BATCH).prefetch(2)

def apply_tta(x,t):
    if t=='identity':return x
    if t=='hflip':return tf.image.flip_left_right(x)
    if t=='vflip':return tf.image.flip_up_down(x)
    if t in ['rot90','rot180','rot270']:return tf.image.rot90(x,{'rot90':1,'rot180':2,'rot270':3}[t])
    raise ValueError('Unknown frozen TTA: '+t)

def load_feature_split(split):
    path=PIPELINE_ROOT/'fusion_features_multiseed_rep_v2'/(split+'_features.npz')
    with np.load(path) as z:
        if not np.array_equal(z['labels'],SPLITS[split].label):raise ValueError('Feature/split labels differ')
        sf=z['simclr_features'].astype(np.float32);rf=z['rotation_features'].astype(np.float32)
    def l2(a):return a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-12)
    return l2(sf),l2(rf)

def predict_heads(sf,rf,rep):
    outputs={};r=PIPELINE_ROOT
    for name,build,args,path,data in [
        ('FusionConcat',build_fusionconcat_model,(sf.shape[1]+rf.shape[1],),r/'fusion_concat_multiseed_rep_v2/fusionconcat_best.weights.h5',np.concatenate([sf,rf],1)),
        ('FusionGatedV2',build_fusiongated_v2_model,(sf.shape[1],rf.shape[1]),r/'fusion_gated_v2_multiseed_rep_v2/fusiongated_v2_best.weights.h5',[sf,rf]),
        ('FusionGatedV3',build_fusiongated_v3_model,(sf.shape[1],rf.shape[1]),r/'fusion_gated_v3_multiseed_rep_v2/fusiongated_best_v3.weights.h5',[sf,rf])]:
        keras.backend.clear_session();gc.collect();model=build(*args);model.load_weights(path);model.trainable=False
        outputs[name]=normp(model.predict(data,batch_size=256,verbose=0));del model
    rd=r/'residual_simclr_rotation_fusion_v1';cfg=json.loads((rd/'config.json').read_text());probs=[]
    for seed in cfg['seeds']:
        keras.backend.clear_session();gc.collect();model=build_residual_fusion_model(sf.shape[1],rf.shape[1]);model.load_weights(rd/f'seed_{seed}/best_by_val_bacc.weights.h5');model.trainable=False
        probs.append(normp(model.predict([sf,rf,np.log(np.clip(rep,1e-7,1)).astype(np.float32)],batch_size=256,verbose=0)));del model
    outputs['ResidualFusion']=normp(np.mean(probs,0));return outputs

def training_results():
    r=PIPELINE_ROOT/'simclr_finetune_multiseed_bacc_v2';cfg=json.loads((r/'multiseed_config.json').read_text())
    feature_info=json.loads((PIPELINE_ROOT/'fusion_features_multiseed_rep_v2/feature_info.json').read_text())
    m=re.search(r'seed_(\d+)',feature_info.get('simclr_finetune_model_path',''))
    if not m:raise ValueError('feature_info.json does not identify the original representative SimCLR seed; cannot infer a replacement safely')
    repseed=int(m.group(1));save_json({'representative_seed':repseed,'source':'feature_info.json.simclr_finetune_model_path','training_predictions':'in_sample; excluded from evaluation metrics'},ROOT/'00_AUDIT'/'training_inference_protocol.json')
    with np.load(r/f'seed_{repseed}/split_probabilities.npz') as z:
        reptest=align_npz(z,'test_prob','y_test',SPLITS['test']);reptrain=align_npz(z,'train_prob','y_train',SPLITS['train'])
    sf,rf=load_feature_split('test');check=predict_heads(sf,rf,reptest);checks=[]
    for name,p in check.items():
        delta=float(np.abs(p-INTERNAL_SOURCES['test'][name]).max());checks.append({'model':name,'max_abs_error':delta})
    pd.DataFrame(checks).to_csv(ROOT/'00_AUDIT'/'fusion_reconstruction_checks.csv',index=False)
    if max(x['max_abs_error'] for x in checks)>2e-5:raise ValueError('Fusion reconstruction failed; training output would not represent saved models')
    with np.load(r/'multiseed_probabilities_no_tta.npz') as z:pno=align_npz(z,'train_prob','y_train',SPLITS['train'])
    ds=make_ds(SPLITS['train']);tta=json.loads((r/'tta_selection/selected_tta_policy.json').read_text())['transforms'];tt=[]
    checkpoint=ROOT/'02_GENERATED_CACHES';checkpoint.mkdir(exist_ok=True)
    for seed in cfg['seeds']:
        print('Training-image TTA inference, seed',seed,flush=True)
        keras.backend.clear_session();gc.collect();model,_=build_simclr_finetune_model();model.load_weights(r/f'seed_{seed}/best_by_val_bacc.weights.h5');model.trainable=False
        batches=[]
        for batch,(x,y) in enumerate(ds):
            batches.append(np.mean([np.asarray(model(apply_tta(x,t),training=False),np.float32) for t in tta],0))
            if batch%200==0:print(' batch',batch,flush=True)
        p=normp(np.concatenate(batches));tt.append(p);np.savez_compressed(checkpoint/f'train_tta_seed_{seed}.npz',image_ids=SPLITS['train'].image_id.to_numpy(dtype=str),probabilities=p)
        del model
    keras.backend.clear_session();gc.collect();model=build_rotation_finetune_model();model.load_weights(PIPELINE_ROOT/'extended_imagenet_rotation_finetune/imagenet_rotation_finetune_best.weights.h5');model.trainable=False
    prot=normp(np.concatenate([np.asarray(model(x,training=False),np.float32) for x,y in ds]));del model
    sf,rf=load_feature_split('train');heads=predict_heads(sf,rf,reptrain)
    sources=OrderedDict(SimCLR_NoTTA=pno,SimCLR_SelectedTTA=normp(np.mean(tt,0)),Rotation=prot)
    sources.update(heads);export_dataset(SPLITS['train'],calibrated_ensemble(sources),base_names=MODEL_NAMES)

def pretext_results():
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix
    d=SPLITS['test'];out=ROOT/'03_SSL_PRETEXT_EVIDENCE';out.mkdir(exist_ok=True)
    keras.backend.clear_session();gc.collect();model,encoder=build_rotation_ssl_model();model.load_weights(PIPELINE_ROOT/'rotation_ssl/rotation_ssl_model_best.weights.h5');model.trainable=False
    rows=[];ds=make_ds(d);offset=0
    for x,y in ds:
        for k in range(4):
            # Original Rotation SSL consumes RGB [0,1], unlike downstream ResNet preprocessing.
            p=normp(model(tf.image.rot90(x,k),training=False).numpy())
            for j,prob in enumerate(p):rows.append({'image_id':d.image_id.iloc[offset+j],'rotation_id':k,'angle_degrees':90*k,'prediction':int(prob.argmax()),'correct':int(prob.argmax()==k),**{f'prob_rotation_{c}':float(prob[c]) for c in range(4)}})
        offset+=len(x)
    rot=pd.DataFrame(rows);rot.to_csv(out/'rotation_pretext_probabilities.csv',index=False)
    if len(rot)!=len(d)*4:raise ValueError('Rotation coverage incomplete')
    pd.DataFrame(confusion_matrix(rot.rotation_id,rot.prediction,labels=range(4)),index=[0,90,180,270],columns=[0,90,180,270]).to_csv(out/'rotation_pretext_confusion_matrix.csv')
    save_json({'images':len(d),'rotated_inputs':len(rot),'accuracy':float(rot.correct.mean()),'BN_mode':'frozen_inference','input':'RGB [0,1]'},out/'rotation_pretext_metrics.json')
    del model,encoder;keras.backend.clear_session();gc.collect()
    encoder,projector=build_simclr_pretrain_models();encoder.load_weights(PIPELINE_ROOT/'simclr_branch/pretrain/simclr_encoder_best.weights.h5');projector.load_weights(PIPELINE_ROOT/'simclr_branch/pretrain/simclr_projector_best.weights.h5');encoder.trainable=False;projector.trainable=False
    rows=[];offset=0;z1s=[];z2s=[];f1s=[];f2s=[]
    for x,y in ds:
        a=tf.stack([simclr_view(im,[SEED,2*(offset+j)+1]) for j,im in enumerate(x)])
        b=tf.stack([simclr_view(im,[SEED,2*(offset+j)+2]) for j,im in enumerate(x)])
        f1=encoder(resnet_preprocess_tensor(a),training=False).numpy();f2=encoder(resnet_preprocess_tensor(b),training=False).numpy()
        z1=projector(f1,training=False).numpy();z2=projector(f2,training=False).numpy();z1s.append(z1);z2s.append(z2);f1s.append(f1);f2s.append(f2)
        def cosine(a,b):return np.sum(a*b,1)/np.maximum(np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1),1e-12)
        for j,(cf,cz) in enumerate(zip(cosine(f1,f2),cosine(z1,z2))):rows.append({'image_id':d.image_id.iloc[offset+j],'encoder_cosine':float(cf),'projector_cosine':float(cz),'view1_seed':f'{SEED},{2*(offset+j)+1}','view2_seed':f'{SEED},{2*(offset+j)+2}'})
        offset+=len(x)
    pair=pd.DataFrame(rows);pair.to_csv(out/'simclr_positive_pair_statistics.csv',index=False)
    np.savez_compressed(out/'simclr_pair_projection_vectors.npz',image_ids=d.image_id.to_numpy(dtype=str),view1=np.concatenate(z1s),view2=np.concatenate(z2s),encoder_view1=np.concatenate(f1s),encoder_view2=np.concatenate(f2s))
    pair[['encoder_cosine','projector_cosine']].describe().to_csv(out/'simclr_pair_distribution.csv')
    save_json({'views':'Stateless horizontal/vertical flips and quarter turns, matching the evidence helper in supplied result-generator. Diagnostic views, not a claim of exact pretraining augmentations.','seed':SEED,'coverage':len(pair),'weights':'SSL encoder and projector; frozen inference'},out/'pair_protocol.json')
    fig,ax=plt.subplots(figsize=(7,4));ax.hist(pair.encoder_cosine,bins=40,alpha=.6,label='Encoder');ax.hist(pair.projector_cosine,bins=40,alpha=.6,label='Projector');ax.set(xlabel='Positive-pair cosine similarity',ylabel='Images');ax.legend();fig.tight_layout();fig.savefig(out/'positive_pair_distribution.png',dpi=200);plt.close(fig)
    del encoder,projector;keras.backend.clear_session();gc.collect()

def full_test_tsne():
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    out=ROOT/'04_FULL_TEST_EMBEDDINGS';out.mkdir(exist_ok=True)
    with np.load(PIPELINE_ROOT/'fusion_features_multiseed_rep_v2/test_features.npz') as z:
        if not np.array_equal(z['labels'],SPLITS['test'].label):raise ValueError('t-SNE labels differ')
        for name,key in [('SimCLR','simclr_features'),('Rotation','rotation_features')]:
            print('Full test t-SNE:',name,flush=True)
            a=StandardScaler().fit_transform(z[key]);a=PCA(n_components=50,random_state=SEED).fit_transform(a)
            xy=TSNE(n_components=2,perplexity=30,init='pca',learning_rate='auto',random_state=SEED).fit_transform(a)
            d=SPLITS['test'][['image_id','label','class_name']].copy();d['x']=xy[:,0];d['y']=xy[:,1];d.to_csv(out/(name+'_all_test_tsne.csv'),index=False)
            fig,ax=plt.subplots(figsize=(8,6))
            for i,c in enumerate(CLASS_NAMES):
                mask=d.label==i;ax.scatter(d.loc[mask,'x'],d.loc[mask,'y'],s=7,label=c,alpha=.6)
            ax.legend(markerscale=2);ax.set_title(f'{name}: all {len(d)} internal-test images');fig.tight_layout();fig.savefig(out/(name+'_all_test_tsne.png'),dpi=200);plt.close(fig)
    save_json({'scope':'test only, 2534 images per branch','scaling':'StandardScaler fitted on test for descriptive visualization only','PCA_dimensions':50,'perplexity':30,'seed':SEED,'not_used_for_model_selection':True},out/'embedding_protocol.json')

def ham_baseline_results():
    # Original HAM notebook evaluated this baseline but did not persist P_BASELINE.
    file=find_one('base_probability_cache/all_seven_sources.npz')
    with np.load(file,allow_pickle=True) as z:
        frame=frame_of(pd.DataFrame({'image_id':z['image_ids'].astype(str),'label':z['labels']}),'HAM10000_SupervisedBaseline','overlap_unverified_secondary_analysis')
    paths=[resolve_image(row) for _,row in frame.iterrows()]
    if any(p is None for p in paths):raise FileNotFoundError('Attach HAM10000 images to recover the unsaved supervised-baseline HAM predictions.')
    frame['filepath']=[str(p) for p in paths];probs=[]
    for seed in [21,35,42,52,62]:
        weight=find_one(f'mbec_plain_supervised_resnet50/models/seed_{seed}/best_by_val_bacc.weights.h5')
        print('Recovering HAM supervised baseline, seed',seed,flush=True)
        keras.backend.clear_session();gc.collect();tf.keras.utils.set_random_seed(seed)
        inp=keras.Input((224,224,3));x=layers.Lambda(resnet_preprocess_tensor)(inp)
        backbone=ResNet50(include_top=False,weights=None,input_shape=(224,224,3),pooling='avg')
        x=backbone(x,training=False);x=layers.Dropout(.30)(x);out=layers.Dense(8,activation='softmax',dtype='float32')(x)
        model=keras.Model(inp,out);model.load_weights(weight);model.trainable=False
        p=normp(np.concatenate([np.asarray(model(x,training=False),np.float32) for x,y in make_ds(frame)]));probs.append(p);del model,backbone
    export_dataset(frame,OrderedDict(SupervisedResNet50_5Seed=normp(np.mean(probs,0))),final='no_designated_final')


## Automatic completion and ZIP export

In [4]:
def copy_existing_evidence():
    # These numerical artifacts are already complete; preserve their extra branch/gate/decomposition columns.
    wanted=['all_internal_test_image_results.csv','feature_vector_statistics_per_image.csv','fusion_gate_values_internal_test.csv','residual_fusion_seed_details.csv','per_image_ensemble_contributions.csv','temperature_and_weight_configuration.csv','class_bias_values.csv','ensemble_reconstruction_check.csv']
    out=ROOT/'05_REUSED_NUMERICAL_EVIDENCE';out.mkdir(exist_ok=True)
    for name in wanted:
        hits=[p for p in FILES if p.name==name and 'skin_lesion_complete_evidence_package' in p.parts]
        if hits:
            p=sorted(hits,key=lambda x:(len(x.parts),str(x)))[0];shutil.copy2(p,out/name)
            PROVENANCE.append({'path':str(p),'sha256':sha(p),'role':'existing numerical evidence, unchanged'})

def stage(name,fn,enabled=True):
    if not enabled:
        STATUS.append({'stage':name,'status':'BLOCKED','reason':'Prerequisite failed'});return False
    print('\n'+name,flush=True)
    try:
        fn();STATUS.append({'stage':name,'status':'PASS'});return True
    except Exception as e:
        err=traceback.format_exc();print(err,flush=True)
        STATUS.append({'stage':name,'status':'FAILED','reason':str(e),'traceback':err});return False

def finish():
    if COVERAGE:pd.DataFrame(COVERAGE).to_csv(ROOT/'coverage.csv',index=False)
    if METRICS:pd.DataFrame(METRICS).to_csv(ROOT/'recomputed_metrics.csv',index=False)
    save_json(PROVENANCE,ROOT/'00_AUDIT'/'input_provenance.json');save_json(STATUS,ROOT/'run_status.json')
    failed=any(r['status'] in ['FAILED','BLOCKED'] for r in STATUS)
    photos_complete=bool(COVERAGE) and all(r['all_images_displayed'] for r in COVERAGE)
    state='INCOMPLETE' if failed else ('COMPLETE_NUMERICAL_IMAGE_ATTACHMENTS_MISSING' if not photos_complete else 'COMPLETE')
    save_json({'status':state,'all_requested_stages_passed':not failed,'all_report_photographs_available':photos_complete,'xai':'EXCLUDED_BY_REQUEST','thesis_submission_archive':'This is a non-XAI result supplement, not the unavailable final submission archive builder.'},ROOT/'COMPLETION.json')
    readme='''# Skin Lesion Per-Image Results — No XAI

Open index.html offline. Every exported dataset has all_image_results.csv, probabilities.npz,
one JSON and one HTML report per image, and confusion matrices recomputed from labelled rows.
The numerical record includes per-model probabilities, predicted class, confidence, margin,
entropy (natural logarithm), true-class probability and competition rank, correctness, agreement
and case category. Unlabelled correctness and true-class statistics are blank.

Training records are in-sample and excluded from recomputed_metrics.csv. Validation records
are model-selection evidence, not an unbiased held-out estimate. Stored validation ensemble
probabilities are preserved (including out-of-fold stacking); the fitted stacker is NOT used
on validation to replace them. Fixed-class macro F1 and balanced accuracy over represented
classes have distinct column names. Full eight-class AUC is blank when a class is absent.

HAM10000 is an overlap-unverified secondary evaluation. Full-string ID mismatch cannot prove
that renamed images are different. This notebook does not invent an original-to-resized ID map.
The separate grouped-protocol DINOv2 results are never pooled with original-split metrics.
Original per-model NPZ caches without IDs are legacy positional artifacts: labels and counts
are checked, but labels alone cannot prove identity within a class. See input_provenance.json.

03_SSL_PRETEXT_EVIDENCE contains full-test rotation and deterministic positive-pair evidence.
04_FULL_TEST_EMBEDDINGS contains coordinates and plots for every internal test image, not 2000.
The new diagnostic probes freeze BatchNorm and use the pretext-specific input preprocessing.
Rotation inputs are RGB [0,1]; SimCLR inputs receive ResNet50 preprocessing. The exact saved
feature-space checkpoints must pass numerical reconstruction before training predictions run.

No Grad-CAM, integrated gradients, saliency, occlusion, or pixel-level XAI is run or copied.
Missing or ambiguous photographs are visibly marked in the HTML and coverage.csv; numerical
results are retained. Attach the original ISIC official-test, PAD, and HAM image inputs to obtain
photographs for all external reports. Original resized internal ISIC downloads automatically
from the same Hugging Face URL used by the supplied training notebook if needed.

This notebook has not retrained models or reselected hyperparameters. It does not create a
thesis PDF, reference-paper collection, or final guideline submission archive. Those are separate
existing deliverables. ZIP is for transport: extract it to browse offline. File hashes are listed
in file_manifest.csv. Read COMPLETION.json and run_status.json before using results.
'''
    (ROOT/'README.md').write_text(readme,encoding='utf-8')
    links=''.join(f'<li><a href="{html.escape(path)}">{html.escape(name)}</a></li>' for name,path in GALLERIES)
    (ROOT/'index.html').write_text(f'<!doctype html><meta charset="utf-8"><style>body{{font:17px Arial;margin:40px;max-width:1000px}}li{{margin:12px}}</style><h1>Skin lesion per-image results — No XAI</h1><p>Status: <b>{state}</b></p><p><a href="coverage.csv">Coverage</a> | <a href="recomputed_metrics.csv">Metrics</a> | <a href="run_status.json">Run status</a> | <a href="README.md">Readme</a></p><ul>{links}</ul>',encoding='utf-8')
    manifest=[{'file':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'sha256':sha(p)} for p in sorted(ROOT.rglob('*')) if p.is_file() and p.name!='file_manifest.csv']
    pd.DataFrame(manifest).to_csv(ROOT/'file_manifest.csv',index=False)
    target=WORK/('Skin_Lesion_Per_Image_No_XAI.zip' if state=='COMPLETE' else 'Skin_Lesion_Per_Image_No_XAI_'+state+'.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
        for p in ROOT.rglob('*'):
            if p.is_file():z.write(p,Path(ROOT.name)/p.relative_to(ROOT))
    print('\nSTATUS:',state,'\nZIP:',target,'\nExtract ZIP, then open',ROOT.name+'/index.html',flush=True)
    display(pd.DataFrame(COVERAGE));display(FileLink(str(target)))
    # Relative /files links work in interactive Kaggle. Saved versions also expose the ZIP in Output.
    from IPython.display import HTML
    display(HTML(f'<a href="/files/{html.escape(target.name)}" download>Download result ZIP</a>'))

def optional_stage(name,fn,required_suffixes):
    missing=[s for s in required_suffixes if not any(p.as_posix().endswith(s) for p in FILES)]
    if missing:
        STATUS.append({'stage':name,'status':'SKIPPED_OPTIONAL','reason':'Optional saved output not attached','missing':missing})
        print('\n'+name+': skipped (optional output not attached)',flush=True)
        return True
    return stage(name,fn)


## Initialize the GPU environment
No TensorFlow/CUDA package replacement or kernel restart is performed. CPU result recovery can still run if GPU setup fails.

In [5]:
GPU_READY = stage('GPU initialization', init_gpu)


GPU initialization
TensorFlow 2.20.0 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Saved-model architectures
These inference-only builders are carried over from the attached HAM evaluation notebook; full checkpoint loads use the original layer structures.

In [6]:
if GPU_READY:
    # ============================================================
    # EXTERNAL CELL E04: INFERENCE-ONLY MODEL ARCHITECTURES
    # ============================================================

    SIMCLR_DROPOUT = 0.30
    ROTATION_FT_DROPOUT = 0.30

    FUSIONCONCAT_DROPOUT = 0.40
    FUSIONCONCAT_HIDDEN_DIM_1 = 512
    FUSIONCONCAT_HIDDEN_DIM_2 = 256
    FUSIONCONCAT_WEIGHT_DECAY = 1e-4

    FUSIONGATED_V2_DROPOUT = 0.50
    FUSIONGATED_V2_WEIGHT_DECAY = 3e-4
    FUSIONGATED_V2_BRANCH_DIM = 256
    FUSIONGATED_V2_GATE_HIDDEN = 128
    FUSIONGATED_V2_HEAD_HIDDEN = 128
    FUSIONGATED_V2_TEMPERATURE = 1.5

    BESTV3_BRANCH_DIM = 256
    BESTV3_DROPOUT_1 = 0.50
    BESTV3_DROPOUT_2 = 0.40
    BESTV3_WEIGHT_DECAY = 4e-4
    BESTV3_GATE_HIDDEN = 128
    BESTV3_HEAD_HIDDEN_1 = 256
    BESTV3_HEAD_HIDDEN_2 = 128
    BESTV3_TEMPERATURE = 1.0

    RESIDUAL_FUSION_BRANCH_DIM = 256
    RESIDUAL_FUSION_HIDDEN_DIM = 256
    RESIDUAL_FUSION_WEIGHT_DECAY = 3e-4
    RESIDUAL_FUSION_DROPOUT = 0.40
    RESIDUAL_FUSION_MAX_ALPHA = 0.35


    def resnet_preprocess_tensor(images):
        images = tf.cast(
            images,
            tf.float32,
        ) * 255.0

        return tf.keras.applications.resnet50.preprocess_input(
            images
        )


    def build_resnet50_encoder(
        name,
        training_flag=False,
    ):
        inputs = keras.Input(
            shape=(
                IMG_SIZE,
                IMG_SIZE,
                3,
            ),
            name=f"{name}_input",
        )

        backbone = ResNet50(
            include_top=False,
            weights=None,
            input_shape=(
                IMG_SIZE,
                IMG_SIZE,
                3,
            ),
            pooling="avg",
        )

        features = backbone(
            inputs,
            training=training_flag,
        )

        return keras.Model(
            inputs,
            features,
            name=name,
        )


    def build_simclr_finetune_model():
        encoder = build_resnet50_encoder(
            "simclr_resnet50_encoder",
            training_flag=False,
        )

        inputs = keras.Input(
            shape=(
                IMG_SIZE,
                IMG_SIZE,
                3,
            ),
            name="finetune_img",
        )

        x = layers.Lambda(
            resnet_preprocess_tensor,
            name="resnet_preprocess",
        )(inputs)

        x = encoder(
            x,
            training=False,
        )

        x = layers.Dropout(
            SIMCLR_DROPOUT,
            name="finetune_dropout",
        )(x)

        outputs = layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            dtype="float32",
            name="finetune_classifier",
        )(x)

        model = keras.Model(
            inputs,
            outputs,
            name="simclr_finetune_model",
        )

        return model, encoder


    def build_rotation_finetune_model():
        encoder = build_resnet50_encoder(
            "rotation_ft_resnet50_encoder",
            training_flag=True,
        )
        encoder.trainable = True

        inputs = keras.Input(
            shape=(
                IMG_SIZE,
                IMG_SIZE,
                3,
            ),
            name="imagenet_rotation_input",
        )

        x = layers.Lambda(
            resnet_preprocess_tensor,
            name="resnet50_preprocess",
        )(inputs)

        # This reproduces the saved architecture.
        x = encoder(
            x,
            training=True
            if encoder.trainable
            else False,
        )

        x = layers.Dropout(
            ROTATION_FT_DROPOUT,
            name="imagenet_rotation_dropout",
        )(x)

        outputs = layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            dtype="float32",
            name="imagenet_rotation_classifier",
        )(x)

        model = keras.Model(
            inputs,
            outputs,
            name="imagenet_rotation_finetune_model",
        )

        return model


    def l2_normalize_features(
        features,
        epsilon=1e-12,
    ):
        features = np.asarray(
            features,
            dtype=np.float32,
        )

        denominator = np.linalg.norm(
            features,
            axis=1,
            keepdims=True,
        )

        return (
            features
            / np.clip(
                denominator,
                epsilon,
                None,
            )
        ).astype(np.float32)


    def build_fusionconcat_model(
        input_dim,
    ):
        inputs = keras.Input(
            shape=(input_dim,),
            name="fusionconcat_input",
        )

        x = layers.BatchNormalization(
            name="bn_input"
        )(inputs)

        x = layers.Dense(
            FUSIONCONCAT_HIDDEN_DIM_1,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    FUSIONCONCAT_WEIGHT_DECAY
                )
            ),
            name="fusionconcat_dense_1",
        )(x)

        x = layers.BatchNormalization(
            name="bn_1"
        )(x)
        x = layers.Activation(
            "relu",
            name="relu_1",
        )(x)
        x = layers.Dropout(
            FUSIONCONCAT_DROPOUT,
            name="dropout_1",
        )(x)

        x = layers.Dense(
            FUSIONCONCAT_HIDDEN_DIM_2,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    FUSIONCONCAT_WEIGHT_DECAY
                )
            ),
            name="fusionconcat_dense_2",
        )(x)

        x = layers.BatchNormalization(
            name="bn_2"
        )(x)
        x = layers.Activation(
            "relu",
            name="relu_2",
        )(x)
        x = layers.Dropout(
            FUSIONCONCAT_DROPOUT,
            name="dropout_2",
        )(x)

        outputs = layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            dtype="float32",
            name="fusionconcat_classifier",
        )(x)

        return keras.Model(
            inputs,
            outputs,
            name="fusionconcat_model",
        )


    def stable_softmax_temperature(
        values,
        temperature,
    ):
        return tf.nn.softmax(
            values / temperature,
            axis=-1,
        )


    def build_fusiongated_v2_model(
        simclr_dim,
        rotation_dim,
    ):
        simclr_input = keras.Input(
            shape=(simclr_dim,),
            name="simclr_input",
        )
        rotation_input = keras.Input(
            shape=(rotation_dim,),
            name="rotation_input",
        )

        simclr_norm = layers.LayerNormalization(
            name="ln_simclr_in"
        )(simclr_input)

        rotation_norm = layers.LayerNormalization(
            name="ln_rotation_in"
        )(rotation_input)

        simclr_projection = layers.Dense(
            FUSIONGATED_V2_BRANCH_DIM,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    FUSIONGATED_V2_WEIGHT_DECAY
                )
            ),
            name="simclr_proj",
        )(simclr_norm)

        simclr_projection = layers.LayerNormalization(
            name="ln_simclr_proj"
        )(simclr_projection)
        simclr_projection = layers.Activation(
            "relu",
            name="relu_simclr_proj",
        )(simclr_projection)

        rotation_projection = layers.Dense(
            FUSIONGATED_V2_BRANCH_DIM,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    FUSIONGATED_V2_WEIGHT_DECAY
                )
            ),
            name="rotation_proj",
        )(rotation_norm)

        rotation_projection = layers.LayerNormalization(
            name="ln_rotation_proj"
        )(rotation_projection)
        rotation_projection = layers.Activation(
            "relu",
            name="relu_rotation_proj",
        )(rotation_projection)

        gate_input = layers.Concatenate(
            name="gate_input_concat"
        )(
            [
                simclr_projection,
                rotation_projection,
            ]
        )

        gate_hidden = layers.Dense(
            FUSIONGATED_V2_GATE_HIDDEN,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    FUSIONGATED_V2_WEIGHT_DECAY
                )
            ),
            name="gate_dense_1",
        )(gate_input)

        gate_hidden = layers.LayerNormalization(
            name="ln_gate_1"
        )(gate_hidden)
        gate_hidden = layers.Activation(
            "relu",
            name="relu_gate_1",
        )(gate_hidden)
        gate_hidden = layers.Dropout(
            FUSIONGATED_V2_DROPOUT,
            name="dropout_gate_1",
        )(gate_hidden)

        gate_logits = layers.Dense(
            2,
            activation=None,
            bias_initializer=(
                keras.initializers.Constant(
                    [0.30, 0.00]
                )
            ),
            name="gate_logits",
        )(gate_hidden)

        gate_weights = layers.Lambda(
            lambda values: (
                stable_softmax_temperature(
                    values,
                    FUSIONGATED_V2_TEMPERATURE,
                )
            ),
            name="branch_gate_softmax",
        )(gate_logits)

        alpha = layers.Lambda(
            lambda values: values[:, 0:1],
            name="alpha_weight",
        )(gate_weights)

        beta = layers.Lambda(
            lambda values: values[:, 1:2],
            name="beta_weight",
        )(gate_weights)

        weighted_simclr = layers.Multiply(
            name="weighted_simclr"
        )(
            [
                simclr_projection,
                alpha,
            ]
        )

        weighted_rotation = layers.Multiply(
            name="weighted_rotation"
        )(
            [
                rotation_projection,
                beta,
            ]
        )

        weighted_sum = layers.Add(
            name="weighted_sum"
        )(
            [
                weighted_simclr,
                weighted_rotation,
            ]
        )

        fused = layers.Concatenate(
            name="residual_fused_concat"
        )(
            [
                weighted_sum,
                simclr_projection,
                rotation_projection,
            ]
        )

        x = layers.LayerNormalization(
            name="ln_fused"
        )(fused)

        x = layers.Dense(
            FUSIONGATED_V2_HEAD_HIDDEN,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    FUSIONGATED_V2_WEIGHT_DECAY
                )
            ),
            name="head_dense_1",
        )(x)

        x = layers.LayerNormalization(
            name="ln_head_1"
        )(x)
        x = layers.Activation(
            "relu",
            name="relu_head_1",
        )(x)
        x = layers.Dropout(
            FUSIONGATED_V2_DROPOUT,
            name="dropout_head_1",
        )(x)

        outputs = layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            dtype="float32",
            name="classifier",
        )(x)

        return keras.Model(
            [
                simclr_input,
                rotation_input,
            ],
            outputs,
            name="fusiongated_v2_model",
        )


    def build_fusiongated_v3_model(
        simclr_dim,
        rotation_dim,
    ):
        simclr_input = keras.Input(
            shape=(simclr_dim,),
            name="simclr_input",
        )
        rotation_input = keras.Input(
            shape=(rotation_dim,),
            name="rotation_input",
        )

        simclr_norm = layers.LayerNormalization(
            name="ln_simclr_in"
        )(simclr_input)
        rotation_norm = layers.LayerNormalization(
            name="ln_rotation_in"
        )(rotation_input)

        simclr_projection = layers.Dense(
            BESTV3_BRANCH_DIM,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    BESTV3_WEIGHT_DECAY
                )
            ),
            name="simclr_proj",
        )(simclr_norm)
        simclr_projection = layers.LayerNormalization(
            name="ln_simclr_proj"
        )(simclr_projection)
        simclr_projection = layers.Activation(
            "relu",
            name="relu_simclr_proj",
        )(simclr_projection)

        rotation_projection = layers.Dense(
            BESTV3_BRANCH_DIM,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    BESTV3_WEIGHT_DECAY
                )
            ),
            name="rotation_proj",
        )(rotation_norm)
        rotation_projection = layers.LayerNormalization(
            name="ln_rotation_proj"
        )(rotation_projection)
        rotation_projection = layers.Activation(
            "relu",
            name="relu_rotation_proj",
        )(rotation_projection)

        gate_input = layers.Concatenate(
            name="gate_input_concat"
        )(
            [
                simclr_projection,
                rotation_projection,
            ]
        )

        gate_hidden = layers.Dense(
            BESTV3_GATE_HIDDEN,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    BESTV3_WEIGHT_DECAY
                )
            ),
            name="gate_dense_1",
        )(gate_input)
        gate_hidden = layers.LayerNormalization(
            name="ln_gate_1"
        )(gate_hidden)
        gate_hidden = layers.Activation(
            "relu",
            name="relu_gate_1",
        )(gate_hidden)
        gate_hidden = layers.Dropout(
            BESTV3_DROPOUT_1,
            name="dropout_gate_1",
        )(gate_hidden)

        gate_logits = layers.Dense(
            2,
            activation=None,
            bias_initializer=(
                keras.initializers.Constant(
                    [0.35, 0.00]
                )
            ),
            name="gate_logits",
        )(gate_hidden)

        gate_weights = layers.Lambda(
            lambda values: (
                stable_softmax_temperature(
                    values,
                    BESTV3_TEMPERATURE,
                )
            ),
            name="branch_gate_softmax",
        )(gate_logits)

        alpha = layers.Lambda(
            lambda values: values[:, 0:1],
            name="alpha_weight",
        )(gate_weights)
        beta = layers.Lambda(
            lambda values: values[:, 1:2],
            name="beta_weight",
        )(gate_weights)

        weighted_simclr = layers.Multiply(
            name="weighted_simclr"
        )(
            [
                simclr_projection,
                alpha,
            ]
        )
        weighted_rotation = layers.Multiply(
            name="weighted_rotation"
        )(
            [
                rotation_projection,
                beta,
            ]
        )
        weighted_sum = layers.Add(
            name="weighted_sum"
        )(
            [
                weighted_simclr,
                weighted_rotation,
            ]
        )

        absolute_difference = layers.Lambda(
            lambda values: tf.abs(
                values[0] - values[1]
            ),
            name="abs_diff",
        )(
            [
                simclr_projection,
                rotation_projection,
            ]
        )

        elementwise_product = layers.Multiply(
            name="elem_prod"
        )(
            [
                simclr_projection,
                rotation_projection,
            ]
        )

        fused = layers.Concatenate(
            name="final_fused_concat"
        )(
            [
                weighted_sum,
                simclr_projection,
                rotation_projection,
                absolute_difference,
                elementwise_product,
            ]
        )

        x = layers.LayerNormalization(
            name="ln_fused"
        )(fused)

        x = layers.Dense(
            BESTV3_HEAD_HIDDEN_1,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    BESTV3_WEIGHT_DECAY
                )
            ),
            name="head_dense_1",
        )(x)
        x = layers.LayerNormalization(
            name="ln_head_1"
        )(x)
        x = layers.Activation(
            "relu",
            name="relu_head_1",
        )(x)
        x = layers.Dropout(
            BESTV3_DROPOUT_1,
            name="dropout_head_1",
        )(x)

        x = layers.Dense(
            BESTV3_HEAD_HIDDEN_2,
            activation=None,
            kernel_regularizer=(
                keras.regularizers.l2(
                    BESTV3_WEIGHT_DECAY
                )
            ),
            name="head_dense_2",
        )(x)
        x = layers.LayerNormalization(
            name="ln_head_2"
        )(x)
        x = layers.Activation(
            "relu",
            name="relu_head_2",
        )(x)
        x = layers.Dropout(
            BESTV3_DROPOUT_2,
            name="dropout_head_2",
        )(x)

        outputs = layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            dtype="float32",
            name="classifier",
        )(x)

        return keras.Model(
            [
                simclr_input,
                rotation_input,
            ],
            outputs,
            name="fusiongated_best_v3_model",
        )


    class BoundedResidualLogitFusion(
        layers.Layer
    ):
        def __init__(
            self,
            max_alpha=0.35,
            initial_alpha=0.08,
            **kwargs,
        ):
            super().__init__(**kwargs)
            self.max_alpha = float(
                max_alpha
            )
            ratio = np.clip(
                initial_alpha / max_alpha,
                1e-4,
                1.0 - 1e-4,
            )
            self.initial_logit = float(
                np.log(
                    ratio
                    / (1.0 - ratio)
                )
            )

        def build(
            self,
            input_shape,
        ):
            self.alpha_logit = (
                self.add_weight(
                    name="alpha_logit",
                    shape=(),
                    initializer=(
                        keras.initializers.Constant(
                            self.initial_logit
                        )
                    ),
                    trainable=True,
                )
            )
            super().build(
                input_shape
            )

        def call(
            self,
            inputs,
        ):
            base_logits, correction_logits = inputs
            alpha = (
                self.max_alpha
                * tf.sigmoid(
                    self.alpha_logit
                )
            )
            return (
                base_logits
                + alpha
                * correction_logits
            )

        def get_config(self):
            config = super().get_config()
            config.update(
                {
                    "max_alpha": self.max_alpha,
                }
            )
            return config


    def build_residual_fusion_model(
        simclr_dim,
        rotation_dim,
    ):
        simclr_input = keras.Input(
            shape=(simclr_dim,),
            name="simclr_feature_input",
        )
        rotation_input = keras.Input(
            shape=(rotation_dim,),
            name="rotation_feature_input",
        )
        base_logits_input = keras.Input(
            shape=(NUM_CLASSES,),
            name="base_simclr_logits",
        )

        simclr_projection = (
            layers.LayerNormalization(
                name="sim_ln_in"
            )(simclr_input)
        )
        simclr_projection = layers.Dense(
            RESIDUAL_FUSION_BRANCH_DIM,
            kernel_regularizer=(
                keras.regularizers.l2(
                    RESIDUAL_FUSION_WEIGHT_DECAY
                )
            ),
            name="sim_projection",
        )(simclr_projection)
        simclr_projection = (
            layers.LayerNormalization(
                name="sim_ln_proj"
            )(simclr_projection)
        )
        simclr_projection = layers.Activation(
            "relu",
            name="sim_relu",
        )(simclr_projection)

        rotation_projection = (
            layers.LayerNormalization(
                name="rot_ln_in"
            )(rotation_input)
        )
        rotation_projection = layers.Dense(
            RESIDUAL_FUSION_BRANCH_DIM,
            kernel_regularizer=(
                keras.regularizers.l2(
                    RESIDUAL_FUSION_WEIGHT_DECAY
                )
            ),
            name="rot_projection",
        )(rotation_projection)
        rotation_projection = (
            layers.LayerNormalization(
                name="rot_ln_proj"
            )(rotation_projection)
        )
        rotation_projection = layers.Activation(
            "relu",
            name="rot_relu",
        )(rotation_projection)

        absolute_difference = layers.Lambda(
            lambda values: tf.abs(
                values[0] - values[1]
            ),
            name="abs_difference",
        )(
            [
                simclr_projection,
                rotation_projection,
            ]
        )

        product = layers.Multiply(
            name="elementwise_product"
        )(
            [
                simclr_projection,
                rotation_projection,
            ]
        )

        fused = layers.Concatenate(
            name="residual_feature_concat"
        )(
            [
                simclr_projection,
                rotation_projection,
                absolute_difference,
                product,
            ]
        )
        fused = layers.LayerNormalization(
            name="fusion_ln"
        )(fused)
        fused = layers.Dense(
            RESIDUAL_FUSION_HIDDEN_DIM,
            kernel_regularizer=(
                keras.regularizers.l2(
                    RESIDUAL_FUSION_WEIGHT_DECAY
                )
            ),
            name="correction_dense",
        )(fused)
        fused = layers.LayerNormalization(
            name="correction_ln"
        )(fused)
        fused = layers.Activation(
            "relu",
            name="correction_relu",
        )(fused)
        fused = layers.Dropout(
            RESIDUAL_FUSION_DROPOUT,
            name="correction_dropout",
        )(fused)
        correction_logits = layers.Dense(
            NUM_CLASSES,
            name="correction_logits",
        )(fused)

        final_logits = (
            BoundedResidualLogitFusion(
                max_alpha=(
                    RESIDUAL_FUSION_MAX_ALPHA
                ),
                initial_alpha=0.08,
                name=(
                    "bounded_residual_"
                    "logit_fusion"
                ),
            )(
                [
                    base_logits_input,
                    correction_logits,
                ]
            )
        )

        output = layers.Softmax(
            dtype="float32",
            name="final_probabilities",
        )(final_logits)

        return keras.Model(
            [
                simclr_input,
                rotation_input,
                base_logits_input,
            ],
            output,
            name=(
                "simclr_preserving_"
                "residual_fusion"
            ),
        )

In [7]:
def build_rotation_ssl_model():
        backbone = keras.applications.ResNet50(
            include_top=False,
            weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
        )
        inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
        x = backbone(inputs, training=True)
        x = layers.GlobalAveragePooling2D(name="gap")(x)
        encoder = keras.Model(inputs, x, name="rotation_encoder")
        model_inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
        features = encoder(model_inputs, training=True)
        features = layers.Dropout(0.5, name="rotation_dropout")(features)
        outputs = layers.Dense(
            4,
            activation="softmax",
            dtype="float32",
            name="rotation_head",
        )(features)
        model = keras.Model(model_inputs, outputs, name="rotation_ssl_model")
        return model, encoder

def build_simclr_pretrain_models():
        inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="simclr_input")
        backbone = keras.applications.ResNet50(
            include_top=False,
            weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            pooling="avg",
        )
        features = backbone(inputs, training=True)
        encoder = keras.Model(inputs, features, name="simclr_resnet50_encoder")

        projector_inputs = keras.Input((2048,), name="simclr_proj_input")
        x = layers.Dense(512, activation="relu", name="simclr_proj_dense_1")(
            projector_inputs
        )
        x = layers.Dropout(0.1, name="simclr_proj_dropout_1")(x)
        outputs = layers.Dense(
            128,
            dtype="float32",
            name="simclr_proj_out",
        )(x)
        projector = keras.Model(
            projector_inputs,
            outputs,
            name="simclr_projection_head",
        )
        return encoder, projector

def simclr_view(image, seed_pair):
        image = tf.image.stateless_random_flip_left_right(image, seed=seed_pair)
        image = tf.image.stateless_random_flip_up_down(
            image, seed=[seed_pair[0] + 1, seed_pair[1] + 1]
        )
        rotations = tf.random.stateless_uniform(
            [],
            seed=[seed_pair[0] + 2, seed_pair[1] + 2],
            minval=0,
            maxval=4,
            dtype=tf.int32,
        )
        image = tf.image.rot90(image, rotations)
        return tf.clip_by_value(image, 0.0, 1.0)

In [8]:
def configure_model_paths():
    global MODEL_PATHS_SPEC
    MODEL_PATHS_SPEC = OrderedDict([
        (
            "SimCLR_NoTTA",
            {
                "type": "npz",
                "path": PIPELINE_ROOT
                / "simclr_finetune_multiseed_bacc_v2"
                / "multiseed_probabilities_no_tta.npz",
            },
        ),
        (
            "SimCLR_SelectedTTA",
            {
                "type": "npz",
                "path": PIPELINE_ROOT
                / "simclr_finetune_multiseed_bacc_v2"
                / "tta_selection"
                / "selected_multiseed_tta_probabilities.npz",
            },
        ),
        (
            "Rotation",
            {
                "type": "csv",
                "val": PIPELINE_ROOT
                / "extended_imagenet_rotation_finetune"
                / "imagenet_rotation_val_predictions.csv",
                "test": PIPELINE_ROOT
                / "extended_imagenet_rotation_finetune"
                / "imagenet_rotation_test_predictions.csv",
            },
        ),
        (
            "FusionConcat",
            {
                "type": "csv",
                "val": PIPELINE_ROOT
                / "fusion_concat_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv",
                "test": PIPELINE_ROOT
                / "fusion_concat_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv",
            },
        ),
        (
            "FusionGatedV2",
            {
                "type": "csv",
                "val": PIPELINE_ROOT
                / "fusion_gated_v2_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv",
                "test": PIPELINE_ROOT
                / "fusion_gated_v2_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv",
            },
        ),
        (
            "FusionGatedV3",
            {
                "type": "csv",
                "val": PIPELINE_ROOT
                / "fusion_gated_v3_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv",
                "test": PIPELINE_ROOT
                / "fusion_gated_v3_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv",
            },
        ),
        (
            "ResidualFusion",
            {
                "type": "npz",
                "path": PIPELINE_ROOT
                / "residual_simclr_rotation_fusion_v1"
                / "multiseed_probabilities.npz",
            },
        ),
    ])

## Run the complete non-XAI supplement and export automatically
Each stage logs its outcome. Failures are not reported as successful completion. No cells need manual selection.

In [9]:
READY = stage('Resolve saved inputs and locked splits', prepare_inputs)
if READY:
    configure_model_paths()
    IMAGES_READY = stage('Resolve internal images / original-source download', ensure_internal_images)
    INTERNAL_READY = stage('Internal validation and test per-image results', internal_results)
    stage('Official ISIC and PAD zero-shot/restricted per-image results', external_results)
    optional_stage('HAM10000 cache conversion and frozen ensemble', ham_results, ['mbec_ham10000_all_models/base_probability_cache/all_seven_sources.npz'])
    optional_stage('Feature-level PAD adapted per-image results', pad_adapted_results, ['PAD_ALL_MODELS_ADAPTED/PAD_ALL_ADAPTED_TEST_PROBABILITIES.npz'])
    optional_stage('Recover unsaved HAM supervised-baseline probabilities', ham_baseline_results, ['mbec_ham10000_all_models/base_probability_cache/all_seven_sources.npz','mbec_plain_supervised_resnet50/supervised_resnet50_5seed_probabilities.npz']) if GPU_READY else None
    stage('Confidence-ensemble PAD meta-adapted per-image results', pad_meta_results)
    optional_stage('Separate grouped-protocol DINOv2 results', grouped_dino_results, ['q2_dinov2_final/DINO_COMPLETE.json'])
    optional_stage('Preserve existing rich numerical evidence', copy_existing_evidence, ['skin_lesion_complete_evidence_package/11_IMAGE_LEVEL_MASTER_RESULTS/all_internal_test_image_results.csv'])
    stage('Full internal-test rotation and SimCLR pair evidence', pretext_results, GPU_READY and IMAGES_READY)
    stage('All training-image predictions and reports', training_results, GPU_READY and IMAGES_READY and INTERNAL_READY)
    stage('All internal-test t-SNE coordinates', full_test_tsne)
finish()



Resolve saved inputs and locked splits
Frozen split restored: {'train': 20264, 'val': 2533, 'test': 2534}

Resolve internal images / original-source download

Internal validation and test per-image results
ISIC2019_val reports 2000 / 2533
Exported ISIC2019_val 2533 images, 13 models; photographs 2533
ISIC2019_test reports 2000 / 2534
Exported ISIC2019_test 2534 images, 13 models; photographs 2534

Official ISIC and PAD zero-shot/restricted per-image results
ISIC_2019_Official_Test reports 2000 / 8238
ISIC_2019_Official_Test reports 4000 / 8238
ISIC_2019_Official_Test reports 6000 / 8238
ISIC_2019_Official_Test reports 8000 / 8238
Exported ISIC_2019_Official_Test 8238 images, 10 models; photographs 8238


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics

PAD_UFES_20_External reports 2000 / 2298
Exported PAD_UFES_20_External 2298 images, 10 models; photographs 2298
PAD_UFES20_Restricted6 reports 2000 / 2298
Exported PAD_UFES20_Restricted6 2298 images, 10 models; photographs 2298

HAM10000 cache conversion and frozen ensemble: skipped (optional output not attached)

Feature-level PAD adapted per-image results: skipped (optional output not attached)

Recover unsaved HAM supervised-baseline probabilities: skipped (optional output not attached)

Confidence-ensemble PAD meta-adapted per-image results
Exported PAD_MetaAdapted_validation 345 images, 2 models; photographs 345
Exported PAD_MetaAdapted_test 345 images, 2 models; photographs 345

Separate grouped-protocol DINOv2 results: skipped (optional output not attached)

Preserve existing rich numerical evidence

Full internal-test rotation and SimCLR pair evidence


I0000 00:00:1789230477.276479      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



All training-image predictions and reports


I0000 00:00:1789230636.523862      66 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Training-image TTA inference, seed 21
 batch 0
 batch 200
 batch 400
 batch 600
Training-image TTA inference, seed 35
 batch 0
 batch 200
 batch 400
 batch 600
Training-image TTA inference, seed 42
 batch 0
 batch 200
 batch 400
 batch 600
Training-image TTA inference, seed 52
 batch 0
 batch 200
 batch 400
 batch 600
Training-image TTA inference, seed 62
 batch 0
 batch 200
 batch 400
 batch 600
ISIC2019_train reports 2000 / 20264
ISIC2019_train reports 4000 / 20264
ISIC2019_train reports 6000 / 20264
ISIC2019_train reports 8000 / 20264
ISIC2019_train reports 10000 / 20264
ISIC2019_train reports 12000 / 20264
ISIC2019_train reports 14000 / 20264
ISIC2019_train reports 16000 / 20264
ISIC2019_train reports 18000 / 20264
ISIC2019_train reports 20000 / 20264
Exported ISIC2019_train 20264 images, 11 models; photographs 20264

All internal-test t-SNE coordinates
Full test t-SNE: SimCLR
Full test t-SNE: Rotation

STATUS: COMPLETE 
ZIP: /kaggle/working/Skin_Lesion_Per_Image_No_XAI.zip 
Extrac

                      dataset  images  models  labelled  html_reports  \
0                ISIC2019_val    2533      13      2533          2533   
1               ISIC2019_test    2534      13      2534          2534   
2     ISIC_2019_Official_Test    8238      10      6191          8238   
3        PAD_UFES_20_External    2298      10      2298          2298   
4      PAD_UFES20_Restricted6    2298      10      2298          2298   
5  PAD_MetaAdapted_validation     345       2       345           345   
6        PAD_MetaAdapted_test     345       2       345           345   
7              ISIC2019_train   20264      11     20264         20264   

   images_displayed  all_images_displayed                       usage  
0              2533                  True  validation_model_selection  
1              2534                  True               internal_test  
2              8238                  True         external_evaluation  
3              2298                  True         exte

/kaggle/working/Skin_Lesion_Per_Image_No_XAI.zip

<IPython.core.display.HTML object>